# 02 · Data Cleaning — Data Wrangling, Deduplicação e Data Quality

**Objetivo deste notebook:** aplicar, passo a passo, cada uma das transformações
de limpeza identificadas no profiling (`01_data_profiling.ipynb`), usando as
funções reutilizáveis de `src/data_cleaning.py`, `src/deduplication.py` e
`src/data_quality.py`. Ao final, geramos a base processada
(`data/processed/customers_clean.csv`) e o relatório de qualidade de dados.

Cada etapa mostra um "antes x depois" para deixar claro o efeito de cada
transformação.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.data_loader import load_raw_data
from src.data_cleaning import (
    clean_email, clean_text_columns, convert_dates, convert_numeric_columns,
    handle_missing_values, standardize_column_names, standardize_customer_name,
    standardize_gender, standardize_state, validate_age,
)
from src.deduplication import (
    deduplicate_by_key, deduplication_summary, find_fuzzy_duplicates, remove_full_duplicates,
)
from src.data_quality import data_quality_score, generate_data_quality_report
from src.descriptive_stats import create_customer_segments

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

df = load_raw_data()
df.shape


(5250, 12)

## 1. Padronização dos nomes de colunas

`"Customer ID"` → `customer_id`, `"Total Spent"` → `total_spent`, etc. —
minúsculas, com underscore, sem espaços.


In [2]:
df = standardize_column_names(df)
df.columns.tolist()


['customer_id',
 'customer_name',
 'email',
 'age',
 'gender',
 'city',
 'state',
 'signup_date',
 'income',
 'purchase_count',
 'total_spent',
 'satisfaction_score']

## 2. Tratamento de texto (espaços) e padronização do nome do cliente


In [3]:
antes = df["customer_name"].head(5).tolist()

df = clean_text_columns(df, ["customer_name", "email", "gender", "city", "state"])
df = standardize_customer_name(df)

depois = df["customer_name"].head(5).tolist()
pd.DataFrame({"antes": antes, "depois": depois})


,antes,depois
0,Emilly da Rocha,Emilly Da Rocha
1,Bento Pastor,Bento Pastor
2,Ana Vitória Cirino,Ana Vitória Cirino
3,gael aragão,Gael Aragão
4,Enrico Carvalho,Enrico Carvalho


## 3. Padronização de categorias: gênero e estado

Usamos dicionários de mapeamento (`MAPA_GENERO`, `MAPA_ESTADOS_UF` em
`src/data_cleaning.py`) para consolidar todas as variantes em categorias
canônicas.


In [4]:
generos_antes = sorted(df["gender"].dropna().unique())
df = standardize_gender(df)
generos_depois = sorted(df["gender"].dropna().unique())

print(f"Categorias de gênero ANTES ({len(generos_antes)}): {generos_antes}")
print(f"Categorias de gênero DEPOIS ({len(generos_depois)}): {generos_depois}")
df["gender"].value_counts()


Categorias de gênero ANTES (17): ['F', 'FEMININO', 'Fem', 'Feminino', 'M', 'MASCULINO', 'Masc', 'Masculino', 'N/I', 'Não binário', 'Outro', 'fem', 'female', 'male', 'masc', 'outro', 'prefer not to say']
Categorias de gênero DEPOIS (3): ['feminino', 'masculino', 'outro']


gender
masculino    2478
feminino     2451
outro         321
Name: count, dtype: int64

In [5]:
estados_antes = df["state"].nunique()
df = standardize_state(df)
estados_depois = df["state"].nunique()

print(f"Valores distintos de estado ANTES: {estados_antes}")
print(f"Valores distintos de estado DEPOIS: {estados_depois} (esperado: até 15 UFs simuladas)")
df["state"].value_counts()


Valores distintos de estado ANTES: 59
Valores distintos de estado DEPOIS: 15 (esperado: até 15 UFs simuladas)


state
SP    1174
RJ     604
MG     490
RS     490
BA     330
PR     330
CE     261
SC     257
PE     256
DF     214
GO     207
PA     181
MT     163
AM     150
ES     143
Name: count, dtype: int64

## 4. Normalização de e-mail

Remove espaços e converte para minúsculas; também sinaliza e-mails
estruturalmente inválidos na coluna auxiliar `email_valido`.


In [6]:
df = clean_email(df)
print(f"E-mails válidos: {df['email_valido'].sum()} de {len(df)} ({df['email_valido'].mean()*100:.2f}%)")
df[["email", "email_valido"]].head(5)


E-mails válidos: 5250 de 5250 (100.00%)


,email,email_valido
0,emilly.da.rocha@mail.com,True
1,bento.pastor@correio.net,True
2,ana.vitoria.cirino@webmail.com,True
3,gael.aragao@correio.net,True
4,enrico.carvalho@webmail.com,True


## 5. Conversão de valores numéricos (incluindo formato monetário em R$)

`"4500"`, `"5.200"` e `"R$ 6.300,00"` devem, todos, virar o número
correspondente em ponto flutuante.


In [7]:
df = convert_numeric_columns(df)
df[["income", "purchase_count", "total_spent", "satisfaction_score", "age"]].dtypes


income                float64
purchase_count          Int64
total_spent           float64
satisfaction_score    float64
age                   float64
dtype: object

In [8]:
df[["income", "total_spent"]].describe()


,income,total_spent
count,4942.000000,5250.000000
mean,6562.163294,3679.669333
std,28056.600494,18483.352321
min,877.000000,0.000000
25%,2462.250000,1133.000000
50%,3385.500000,1655.000000
75%,4601.000000,2293.750000
max,392045.000000,247008.000000


## 6. Conversão de datas

Reconhece múltiplos formatos de entrada e cria colunas derivadas (ano, mês,
trimestre, dia da semana), úteis para a análise temporal no notebook 04.


In [9]:
df = convert_dates(df)
print(f"Datas não convertidas (NaT): {df['signup_date'].isna().sum()}")
df[["signup_date", "signup_date_year", "signup_date_month", "signup_date_quarter", "signup_date_weekday"]].head(5)


Datas não convertidas (NaT): 0


,signup_date,signup_date_year,signup_date_month,signup_date_quarter,signup_date_weekday
0,2025-05-30,2025,5,2,Friday
1,2023-03-17,2023,3,1,Friday
2,2023-08-08,2023,8,3,Tuesday
3,2023-08-12,2023,8,3,Saturday
4,2024-06-29,2024,6,2,Saturday


## 7. Validação de idade (18-100)

Idades fora do intervalo **não são removidas silenciosamente** — são
convertidas para ausente (`NaN`) e registradas em relatório, para posterior
imputação na etapa de tratamento de valores ausentes.


In [10]:
df, relatorio_idade = validate_age(df)
relatorio_idade


{'coluna': 'age',
 'regra': '18 <= age <= 100',
 'quantidade_invalida': 77,
 'percentual_invalido': 1.47,
 'estrategia': 'Convertido para ausente (NaN) e imputado pela mediana em handle_missing_values'}

## 8. Deduplicação

Três níveis, do mais seguro ao mais investigativo:

1. Duplicidade completa → removida.
2. Duplicidade por `customer_id` → mantém o registro mais recente
   (`signup_date` mais alto).
3. Duplicidade aproximada (fuzzy matching por nome) → apenas reportada.


In [11]:
linhas_antes_dedup = len(df)

df, removidos_completos = remove_full_duplicates(df)
df, removidos_por_chave = deduplicate_by_key(df, key="customer_id", sort_column="signup_date")

print(f"Linhas antes da deduplicação: {linhas_antes_dedup}")
print(f"Duplicidades completas removidas: {removidos_completos}")
print(f"Duplicidades por customer_id removidas: {removidos_por_chave}")
print(f"Linhas após deduplicação determinística: {len(df)}")
print(f"customer_id únicos: {df['customer_id'].nunique()} (deve ser igual ao nº de linhas: {len(df)})")


Linhas antes da deduplicação: 5250
Duplicidades completas removidas: 60
Duplicidades por customer_id removidas: 120
Linhas após deduplicação determinística: 5070
customer_id únicos: 5070 (deve ser igual ao nº de linhas: 5070)


In [12]:
# Investigação de duplicidade aproximada (fuzzy matching) — NÃO remove nada automaticamente.
candidatos_fuzzy = find_fuzzy_duplicates(df, column="customer_name", threshold=90.0)
print(f"Candidatos a duplicidade aproximada encontrados: {len(candidatos_fuzzy)}")
candidatos_fuzzy.sort_values("similarity_score", ascending=False).head(10)


Candidatos a duplicidade aproximada encontrados: 679


,record_1,record_2,name_1,name_2,similarity_score,possible_duplicate
0,CUST004741,CUST004090,Yasmin Brito,Yasmin Brito,100.0,True
1,CUST000276,CUST005046,Yuri Da Rocha,Yuri Da Rocha,100.0,True
2,CUST004946,CUST001054,Aurora Macedo,Aurora Macedo,100.0,True
3,CUST001834,CUST004168,Amanda Vargas,Amanda Vargas,100.0,True
4,CUST002919,CUST002220,Ana Luiza Fernandes,Ana Luiza Fernandes,100.0,True
5,CUST001021,CUST003276,Sra. Maria Julia Pimenta,Sra. Maria Júlia Pimenta,100.0,True
6,CUST002897,CUST005056,Srta. Vitória Almeida,Srta. Vitória Almeida,100.0,True
7,CUST000107,CUST005059,Stephany Dias,Stephany Dias,100.0,True
8,CUST000363,CUST005002,Sra. Anna Liz Farias,Sra. Anna Liz Farias,100.0,True
9,CUST002364,CUST003774,Theo Nascimento,Theo Nascimento,100.0,True


**Interpretação:** cada linha da tabela acima é um **par candidato** — dois
`customer_id` diferentes com nomes muito parecidos (score de similaridade
≥ 90). Isso pode indicar (a) a mesma pessoa cadastrada duas vezes por engano,
ou (b) uma coincidência de nomes comuns (ex.: "Maria Silva" é um nome
frequente no Brasil). A decisão de mesclar ou não esses registros é de
negócio, não técnica — por isso o relatório é apenas investigativo e é
exportado para `data/output/fuzzy_duplicates_report.csv`, para revisão manual
por um analista.


In [13]:
resumo_dedup = deduplication_summary(linhas_antes_dedup, removidos_completos, removidos_por_chave, len(candidatos_fuzzy))
resumo_dedup


,etapa,quantidade
0,Registros originais (RAW),5250
1,Duplicidades completas removidas,60
2,Duplicidades por customer_id removidas,120
3,Registros após deduplicação determinística,5070
4,Candidatos a duplicidade aproximada (apenas in...,679


## 9. Tratamento de valores ausentes

Estratégia documentada por coluna — nunca `fillna(0)`:

| Coluna | Estratégia | Justificativa |
|---|---|---|
| `age` | mediana global | distribuição aproximadamente simétrica |
| `income` | mediana por estado (`state`), com fallback para mediana global | renda varia sistematicamente por região |
| `satisfaction_score` | mediana global | escala limitada (1-5), pouco sensível a outliers |
| `city` | categoria `"unknown"` | não é seguro inferir a cidade a partir de outras colunas |


In [14]:
df, relatorio_missing = handle_missing_values(df)
relatorio_missing


,coluna,missing_percent,estrategia
0,age,5.42,mediana global (38.0)
1,income,5.84,"mediana por grupo (state), fallback mediana gl..."
2,satisfaction_score,7.89,mediana global (4.0)
3,city,4.93,categoria 'unknown'


In [15]:
print("Valores ausentes remanescentes por coluna:")
df.isna().sum()


Valores ausentes remanescentes por coluna:


customer_id            0
customer_name          0
email                  0
age                    0
gender                 0
city                   0
state                  0
signup_date            0
income                 0
purchase_count         0
total_spent            0
satisfaction_score     0
email_valido           0
signup_date_year       0
signup_date_month      0
signup_date_quarter    0
signup_date_weekday    0
dtype: int64

## 10. Segmentação de clientes

Criamos os segmentos de negócio (`Low / Medium / High Value`) já nesta etapa,
para que fiquem disponíveis na base processada e sejam reutilizados nos
notebooks seguintes.


In [16]:
df = create_customer_segments(df)
df["customer_segment"].value_counts()


customer_segment
Medium Value    1692
Low Value       1690
High Value      1688
Name: count, dtype: int64

## 11. Relatório de Data Quality — antes x depois

Comparamos o *Data Quality Score* calculado sobre a base bruta (apenas com
conversão numérica, para permitir os checks de validade) e sobre a base
processada final.


In [17]:
df_raw_para_dq = convert_numeric_columns(standardize_column_names(load_raw_data()))
relatorio_raw = generate_data_quality_report(df_raw_para_dq)
score_raw = data_quality_score(relatorio_raw)

relatorio_processado = generate_data_quality_report(df)
score_processado = data_quality_score(relatorio_processado)

print(f"Data Quality Score (RAW):        {score_raw:.2f} / 100")
print(f"Data Quality Score (PROCESSADO): {score_processado:.2f} / 100")


Data Quality Score (RAW):        89.39 / 100
Data Quality Score (PROCESSADO): 100.00 / 100


In [18]:
relatorio_raw.sort_values("value").head(10)


,metric,value,status,description
16,consistency_gender_categories,1.07,CRÍTICO,Percentual de registros com categoria de gêner...
17,consistency_state_format,42.3,CRÍTICO,Percentual de registros com UF padronizada em ...
14,validity_satisfaction_score,92.25,CRÍTICO,Percentual de notas de satisfação dentro do in...
11,completeness_satisfaction_score,92.25,ATENÇÃO,Percentual de valores preenchidos na coluna 's...
15,validity_income_positive,94.13,CRÍTICO,Percentual de rendas com valor positivo (> 0).
8,completeness_income,94.13,ATENÇÃO,Percentual de valores preenchidos na coluna 'i...
13,validity_age,94.59,CRÍTICO,Percentual de idades dentro do intervalo válid...
5,completeness_city,95.14,ATENÇÃO,Percentual de valores preenchidos na coluna 'c...
3,completeness_age,96.06,ATENÇÃO,Percentual de valores preenchidos na coluna 'a...
12,uniqueness_customer_id,96.57,CRÍTICO,Percentual de 'customer_id' únicos (5070 de 52...


**Interpretação:** as métricas mais baixas na base RAW são exatamente as que
tratamos neste notebook — `consistency_gender_categories` e
`consistency_state_format` (categorias não padronizadas), `uniqueness_customer_id`
(duplicidades) e `validity_age`/`validity_income_positive`/`validity_satisfaction_score`
(idades inválidas e valores ausentes). Após a limpeza, o score sobe
significativamente, confirmando que as transformações aplicadas resolveram os
problemas reais identificados no profiling — e não apenas problemas
hipotéticos.


## 12. Exportação da base processada


In [19]:
from src.data_loader import save_dataframe, DIR_RAIZ

save_dataframe(df, DIR_RAIZ / "data" / "processed" / "customers_clean.csv")
print(f"Base processada salva com {len(df)} linhas e {df.shape[1]} colunas.")
df.head(5)


Base processada salva com 5070 linhas e 18 colunas.


,customer_id,customer_name,email,age,gender,city,state,signup_date,income,purchase_count,total_spent,satisfaction_score,email_valido,signup_date_year,signup_date_month,signup_date_quarter,signup_date_weekday,customer_segment
0,CUST001040,Maria Vitória Santos,maria.vitoria.santos@mail.com,26.0,feminino,Caruaru,PE,2021-01-25,3481.0,7,1870.0,4.9,True,2021,1,1,Monday,Medium Value
1,CUST000492,Marina Camargo,marina.camargo@correio.net,18.0,feminino,Campinas,SP,2021-02-10,2585.0,5,1546.0,4.7,True,2021,2,1,Wednesday,Medium Value
2,CUST004279,Allana Novaes,allana.novaes@webmail.com,38.0,feminino,Juazeiro do Norte,CE,2021-02-15,5098.0,8,3332.0,4.1,True,2021,2,1,Monday,High Value
3,CUST003069,Bernardo Da Rosa,bernardo.da.rosa@email.com,18.0,masculino,Ribeirão Preto,SP,2021-02-18,3963.0,11,2496.0,5.0,True,2021,2,1,Thursday,High Value
4,CUST003410,Francisco Santos,francisco.santos@webmail.com,33.0,masculino,Várzea Grande,MT,2021-02-25,2610.0,7,990.0,3.9,True,2021,2,1,Thursday,Low Value


## Conclusão

A base processada (`customers_clean.csv`) está pronta para a análise
estatística e exploratória dos próximos notebooks: tipos corretos, categorias
padronizadas, sem duplicidades determinísticas, com valores ausentes tratados
de forma documentada e com segmentos de negócio já atribuídos. Os relatórios
de auditoria (duplicidade aproximada, estratégia de imputação, qualidade de
dados antes/depois) foram exportados para `data/output/`, garantindo
rastreabilidade de cada decisão tomada.
